# Set Library Path

In [4]:
.libPaths("/home/vhaghani/anaconda3/envs/jupyter_nb_R4.4.2/lib/R/library")

# Load Libraries

In [6]:
library(enrichR)
library(dplyr)
library(openxlsx)
library(ggplot2)
library(readxl)
library(tidyr)
library(viridis)

# Load Data

In [7]:
# Load
df <- read.csv("05_DEGs/rescue_genes.csv")
                
# View
head(df)

,gene,rank,Control.1,Control.2,Control.3,HD.1,HD.2,HD.3,HDtx.1,HDtx.2,HDtx.3
,<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,ST8SIA3,1,3.708,3.179,3.332,5.678,5.707,5.749,3.280,3.073,2.874
2,STXBP6,2,3.391,2.964,3.124,5.693,5.207,5.694,3.044,3.109,3.107
3,STXBP5-AS1,3,2.448,1.950,2.125,4.405,4.616,4.381,1.950,1.658,2.103
4,AHI1,4,5.485,5.573,5.663,6.814,6.734,6.708,5.478,5.804,5.841
5,GABRA2,5,3.098,2.964,2.841,5.216,4.967,5.267,2.979,2.880,2.597
6,LUZP2,6,2.986,3.224,2.820,5.165,4.937,4.988,3.186,2.774,2.755


# Gene Ontology

In [15]:
# Perform the enrichR analysis on the gene list
enrichr_results <- enrichr(df$gene, "GO_Biological_Process_2025")
enrichr_results<- enrichr_results$GO_Biological_Process_2025

# View
head(enrichr_results)

Uploading data to Enrichr... Done.
  Querying GO_Biological_Process_2025... Done.
Parsing results... Done.


,Term,Overlap,P.value,Adjusted.P.value,Old.P.value,Old.Adjusted.P.value,Odds.Ratio,Combined.Score,Genes
,<chr>,<chr>,<dbl>,<dbl>,<int>,<int>,<dbl>,<dbl>,<chr>
1,Gamma-Aminobutyric Acid Signaling Pathway (GO:0007214),6/23,3.800956e-08,0.0000294194,0,0,41.121799,702.58355,GABRA2;GABRA1;GABRB1;PLCL1;GABRE;GABRG2
2,Chemical Synaptic Transmission (GO:0007268),12/260,3.448369e-06,0.0013345186,0,0,5.775767,72.64534,GABRA2;CRHBP;GRIA2;GABRA1;HTR7;RIT2;TH;APBA1;PDYN;GABRE;LIN7B;GABRG2
3,"Synaptic Transmission, GABAergic (GO:0051932)",4/16,9.710968e-06,0.0025054298,0,0,38.395349,443.16889,GABRA2;GABRA1;GABRE;GABRG2
4,Inhibitory Synapse Assembly (GO:1904862),4/21,3.085452e-05,0.0059703497,0,0,27.095759,281.42271,GABRA2;GABRA1;GABRE;GABRG2
5,Chloride Transport (GO:0006821),6/90,1.440342e-04,0.0222964952,0,0,8.294118,73.36528,GABRA2;CLIC5;GABRA1;GABRB1;GABRE;GABRG2
6,Chloride Transmembrane Transport (GO:1902476),5/65,2.686955e-04,0.0346617200,0,0,9.631579,79.19018,GABRA2;GABRA1;GABRB1;GABRE;GABRG2


In [19]:
# Filter to significant terms only
df <- enrichr_results[enrichr_results$Adjusted.P.value < 0.05 & !is.na(enrichr_results$Adjusted.P.value), ]

# Determine gene count
df$Count <- as.integer(sub("/.*", "", df$Overlap))

# Calculate gene ratio
df$Gene.Ratio <- sapply(strsplit(df$Overlap, "/"), function(x) as.numeric(x[1]) / as.numeric(x[2]))
                        
# View
head(df)

,Term,Overlap,P.value,Adjusted.P.value,Old.P.value,Old.Adjusted.P.value,Odds.Ratio,Combined.Score,Genes,Count,Gene.Ratio
,<chr>,<chr>,<dbl>,<dbl>,<int>,<int>,<dbl>,<dbl>,<chr>,<int>,<dbl>
1,Gamma-Aminobutyric Acid Signaling Pathway (GO:0007214),6/23,3.800956e-08,0.0000294194,0,0,41.121799,702.58355,GABRA2;GABRA1;GABRB1;PLCL1;GABRE;GABRG2,6,0.26086957
2,Chemical Synaptic Transmission (GO:0007268),12/260,3.448369e-06,0.0013345186,0,0,5.775767,72.64534,GABRA2;CRHBP;GRIA2;GABRA1;HTR7;RIT2;TH;APBA1;PDYN;GABRE;LIN7B;GABRG2,12,0.04615385
3,"Synaptic Transmission, GABAergic (GO:0051932)",4/16,9.710968e-06,0.0025054298,0,0,38.395349,443.16889,GABRA2;GABRA1;GABRE;GABRG2,4,0.25000000
4,Inhibitory Synapse Assembly (GO:1904862),4/21,3.085452e-05,0.0059703497,0,0,27.095759,281.42271,GABRA2;GABRA1;GABRE;GABRG2,4,0.19047619
5,Chloride Transport (GO:0006821),6/90,1.440342e-04,0.0222964952,0,0,8.294118,73.36528,GABRA2;CLIC5;GABRA1;GABRB1;GABRE;GABRG2,6,0.06666667
6,Chloride Transmembrane Transport (GO:1902476),5/65,2.686955e-04,0.0346617200,0,0,9.631579,79.19018,GABRA2;GABRA1;GABRB1;GABRE;GABRG2,5,0.07692308


# Dot Plot

In [20]:
plot_dot_signatures <- function(df, filename,
                                height = 12, width = 11,
                                title_size       = 14,
                                x_title_size     = 12,
                                y_title_size     = 12,
                                x_axis_size      = 6,
                                y_axis_size      = 6,
                                legend_title_size = 10,
                                legend_text_size  = 8,
                                legend_key_size   = 1) {
  # Reorder gene ratios
  df$Term <- factor(df$Term, levels = df$Term[order(df$Gene.Ratio, decreasing = FALSE)])

  # Plot
  dot_plot <- ggplot(df, aes(x = Gene.Ratio, y = Term,
                             size = Count, fill = Adjusted.P.value)) +
    geom_point(shape = 21) +
    scale_fill_viridis() +
    xlab('Gene Ratio') + ylab('Term') +
    labs(title = "GO Biological Process Enrichment Terms\nRescue Genes") +
    theme(
      panel.background = element_rect(fill = "white", color = NA),
      panel.grid.major  = element_blank(),
      panel.grid.minor  = element_blank(),
      panel.border      = element_blank(),
      axis.line         = element_line(color = "black"),
      plot.background   = element_rect(fill = "white", color = NA),

      plot.title        = element_text(size = title_size),
      axis.title.x      = element_text(size = x_title_size),
      axis.title.y      = element_text(size = y_title_size),
      axis.text.x       = element_text(angle = 0, hjust = 0.5, size = x_axis_size),
      axis.text.y       = element_text(size = y_axis_size),

      legend.title      = element_text(size = legend_title_size),
      legend.text       = element_text(size = legend_text_size),
      legend.key.size   = unit(legend_key_size, "cm")
    )

  ggsave(filename = filename, plot = dot_plot, height = height,
         width = width, limitsize = FALSE)

  invisible(dot_plot)
}

In [26]:
# Plot
plot_dot_signatures(
  df       = df,
  filename = "00_FIGURES/rescue_genes_go_bp_dot_plot.pdf",
  height   = 4,
  width    = 7,
  title_size        = 12,
  x_title_size      = 10,
  y_title_size      = 10,
  x_axis_size       = 9,
  y_axis_size       = 9,
  legend_title_size = 8,
  legend_text_size  = 6,
  legend_key_size   = 0.6
)